# <font color="#418FDE" size="6.5" uppercase>**Cleaning Engineering Data**</font>

>Last update: 20260710.
    
By the end of this Lecture, you will be able to:
- Detect missing, duplicated, inconsistent, and physically unrealistic values in engineering datasets. 
- Clean tabular and time-series data using transparent pandas and NumPy operations. 
- Document data-cleaning decisions so model results remain traceable and defensible. 


## **1. Data Quality Checks**

### **1.1. Missing Value Detection**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Mechanical Engineers/Module_02/Lecture_A/image_01_01.jpg?v=1783659001" width="250">



>* Find hidden gaps before analysis or modeling
>* Use system context to judge importance

>* Check missing data amount and patterns
>* Clustered gaps may reveal biased collection

>* Recognize hidden placeholders for missing measurements
>* Use context to document defensible cleaning decisions



In [ ]:
#@title Python Code - Missing Value Detection

# Detect missing values in engineering data.
# Use pandas masks for transparent checks.
# Summarize gaps before cleaning decisions.

import numpy as np
import pandas as pd

# Build a tiny pump sensor dataset.
data = {
    "time_min": [0, 1, 2, 3, 4, 5],
    "pressure_psi": [52.1, np.nan, 51.8, -999, 52.4, 52.2],
    "flow_lpm": [18.0, 18.2, None, 0.0, 18.1, -1.0],
    "operator_note": ["ok", "", "sensor reset", "not available", "ok", "ok"],
}

# Convert the dictionary into a table.
df = pd.DataFrame(data)

# Replace common text placeholders with missing values.
text_missing = ["", "not available"]
df = df.replace(text_missing, np.nan)

# Replace sensor fault codes with missing values.
fault_codes = {"pressure_psi": [-999], "flow_lpm": [-1.0]}
df = df.replace(fault_codes, np.nan)

# Count missing values in each column.
missing_counts = df.isna().sum()
missing_percent = (df.isna().mean() * 100).round(1)

# Find rows containing any missing measurement.
rows_with_gaps = df[df.isna().any(axis=1)]["time_min"].tolist()

# Create a compact summary table.
summary = pd.DataFrame({"missing_count": missing_counts, "missing_percent": missing_percent})

# Print concise results for traceable decisions.
print("Missing value summary by column:")
print(summary.to_string())
print("Rows with any missing value:", rows_with_gaps)
print("Decision note: investigate sensor faults before imputation.")



### **1.2. Duplicate Data**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Mechanical Engineers/Module_02/Lecture_A/image_01_02.jpg?v=1783659003" width="250">



>* Repeated records can come from many sources
>* Duplicates can bias engineering analysis results

>* Define uniqueness using relevant engineering fields
>* Check near-duplicates with engineering judgment

>* Interpret duplicates before removing them
>* Check patterns to protect analysis reliability



In [ ]:
#@title Python Code - Duplicate Data

# Duplicate rows can bias engineering summaries.
# Pandas can flag exact and contextual duplicates.
# Cleaning decisions should remain traceable.

import pandas as pd
import numpy as np

# Create a tiny vibration dataset.
data = {
    "asset": ["Pump 07", "Pump 07", "PUMP-7", "Fan 02", "Fan 02", "Fan 02"],
    "timestamp": ["10:00:00", "10:00:00", "10:00:01", "10:05:00", "10:05:00", "10:10:00"],
    "sensor": ["VIB", "VIB", "VIB", "TEMP", "TEMP", "TEMP"],

    "value": [0.42, 0.42, 0.42, 185.0, 185.0, 188.0],
    "unit": ["in/s", "in/s", "in/s", "F", "F", "F"],
}

# Build the table safely.
df = pd.DataFrame(data)
assert df.shape == (6, 5)

# Flag rows duplicated across every column.
exact_mask = df.duplicated(keep=False)
exact_count = int(exact_mask.sum())

# Standardize asset labels for contextual checking.
clean_asset = df["asset"].str.upper().str.replace(" ", "-", regex=False)
clean_asset = clean_asset.str.replace("PUMP-07", "PUMP-7", regex=False)

# Add normalized fields without overwriting raw data.
df_checked = df.assign(clean_asset=clean_asset)
key_cols = ["clean_asset", "timestamp", "sensor", "value", "unit"]

# Flag duplicates using engineering uniqueness rules.
context_mask = df_checked.duplicated(subset=key_cols, keep=False)
context_count = int(context_mask.sum())

# Keep one record per contextual duplicate group.
cleaned = df_checked.drop_duplicates(subset=key_cols, keep="first")
removed_count = len(df_checked) - len(cleaned)

# Document the cleaning decision compactly.
decision = "Removed repeated asset-time-sensor-value records after label normalization."

# Print concise, traceable results.
print(f"Rows before cleaning: {len(df_checked)}")
print(f"Exact duplicate rows flagged: {exact_count}")
print(f"Contextual duplicate rows flagged: {context_count}")
print(f"Rows removed: {removed_count}")
print(f"Rows after cleaning: {len(cleaned)}")
print(f"Decision: {decision}")



### **1.3. Inconsistent Values**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Mechanical Engineers/Module_02/Lecture_A/image_01_03.jpg?v=1783659006" width="250">



>* Values may conflict in units or formats
>* Inconsistencies can mislead analysis and models

>* Check categories against valid engineering labels
>* Verify numerical units, signs, and resolution

>* Check relationships across fields and time
>* Flag inconsistencies for engineering investigation



In [ ]:
#@title Python Code - Inconsistent Values

# Detect inconsistent engineering values with pandas.
# Compare categories, units, and relationships.
# Flag issues before changing raw data.

# pandas and numpy are preinstalled in Colab.
import pandas as pd
import numpy as np

# Create a tiny pump test dataset.
data = {
    "pump_id": ["P-01", "p01", "P_01", "P-02", "P-02"],
    "status": ["running", "Run", "ON", "closed", "closed"],
    "pressure_kpa": [210, 2.1, 205, 190, 1.9],
    "flow_lpm": [48, 47, 49, 42, 0],
}

# Load the records into pandas.
df = pd.DataFrame(data)

# Standardize pump identifiers for comparison.
df["pump_id_clean"] = (
    df["pump_id"].str.upper().str.replace("_", "-", regex=False)
)

# Map status words into common labels.
status_map = {
    "RUNNING": "RUNNING", "RUN": "RUNNING", "ON": "RUNNING",
    "CLOSED": "CLOSED",
}

# Standardize status values safely.
df["status_clean"] = df["status"].str.upper().map(status_map)

# Flag likely bar values inside kPa column.
df["possible_bar_in_kpa"] = df["pressure_kpa"].between(1, 5)

# Flag closed valves with high flow.
df["closed_with_flow"] = (
    (df["status_clean"] == "CLOSED") & (df["flow_lpm"] > 5)
)

# Count each inconsistency type.
summary = {
    "pump naming variants": df["pump_id_clean"].nunique() < df["pump_id"].nunique(),
    "unmapped statuses": int(df["status_clean"].isna().sum()),
    "possible unit mix": int(df["possible_bar_in_kpa"].sum()),
    "status flow conflicts": int(df["closed_with_flow"].sum()),
}

# Show compact results only.
print("Inconsistent value checks:")
print(summary)

# Show only flagged rows and key columns.
flagged = df[df["possible_bar_in_kpa"] | df["closed_with_flow"]]
print(flagged[["pump_id", "status", "pressure_kpa", "flow_lpm"]].to_string(index=False))



## **2. Core Cleaning Operations**

### **2.1. Unit Standardization**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Mechanical Engineers/Module_02/Lecture_A/image_02_01.jpg?v=1783658999" width="250">



>* Consistent units prevent misleading engineering results
>* Document conversions clearly for transparent workflows

>* Separate numeric values from unit labels
>* Apply systematic pandas and NumPy conversions

>* Use context and engineering judgment for conversions
>* Validate ranges and document conversion assumptions



In [ ]:
#@title Python Code - Unit Standardization

# Unit standardization prevents hidden engineering calculation errors.
# This example keeps raw values for traceability.
# Pandas converts mixed units using explicit rules.

import numpy as np
import pandas as pd

# Create a tiny mixed-unit pump dataset.
raw_data = {
    "test_id": ["A", "B", "C", "D", "E"],
    "flow_value": [120.0, 7.2, 95.0, 0.006, 150.0],

    "flow_unit": ["L/min", "m3/h", "gpm", "m3/s", "L/min"],
    "pressure_value": [250.0, 2.8, 36.0, 310000.0, 4.0],
    "pressure_unit": ["kPa", "bar", "psi", "Pa", "bar"],
}

# Store raw fields before cleaning.
df = pd.DataFrame(raw_data)
raw_columns = df.copy(deep=True)

# Define transparent conversion factors.
flow_to_m3s = {
    "L/min": 1.0 / 60000.0,
    "m3/h": 1.0 / 3600.0,
    "gpm": 0.00378541 / 60.0,
    "m3/s": 1.0,
}

# Define pressure conversions into pascals.
pressure_to_pa = {
    "Pa": 1.0,
    "kPa": 1000.0,
    "bar": 100000.0,
    "psi": 6894.757,
}

# Convert units using mapped factors.
df["flow_m3_s"] = df["flow_value"] * df["flow_unit"].map(flow_to_m3s)
df["pressure_Pa"] = df["pressure_value"] * df["pressure_unit"].map(pressure_to_pa)

# Check for unsupported units after mapping.
missing_flow = df["flow_m3_s"].isna().sum()
missing_pressure = df["pressure_Pa"].isna().sum()

# Flag physically suspicious standardized values.
df["flow_flag"] = np.where(df["flow_m3_s"] > 0.01, "check", "ok")
df["pressure_flag"] = np.where(df["pressure_Pa"] > 500000, "check", "ok")

# Keep a compact cleaned view for review.
clean_view = df[["test_id", "flow_m3_s", "pressure_Pa", "flow_flag", "pressure_flag"]]

# Print concise traceable results only.
print("Target units: flow=m^3/s, pressure=Pa")
print(f"Unsupported flow units: {missing_flow}")
print(f"Unsupported pressure units: {missing_pressure}")
print(clean_view.round(4).to_string(index=False))
print("Raw columns preserved separately:", list(raw_columns.columns))



### **2.2. Outlier Handling**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Mechanical Engineers/Module_02/Lecture_A/image_02_02.jpg?v=1783658996" width="250">



>* Identify unusual values without assuming errors
>* Use transparent rules to treat outliers

>* Start with physical limits and engineering judgment
>* Use statistics with operating context

>* Match outlier treatment to cause and purpose
>* Preserve timing, flags, and cleaning traceability



In [ ]:
#@title Python Code - Outlier Handling

# Demonstrate transparent outlier handling.
# Use small engineering sensor data.
# Preserve original values for traceability.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Create a tiny pressure time series.
time_s = np.arange(0, 12, 1)
pressure_kpa = np.array([101, 102, 103, 500, 104, 105, -20, 106, 107, 108, 109, 110])
flow_lpm = np.array([10, 11, 10, 12, 11, 10, 9, 80, 11, 10, 12, 11])

# Store readings in a dataframe.
df = pd.DataFrame({"time_s": time_s, "pressure_kpa": pressure_kpa, "flow_lpm": flow_lpm})

# Keep original columns before cleaning.
df["pressure_original"] = df["pressure_kpa"]
df["flow_original"] = df["flow_lpm"]

# Flag physically impossible pressure values.
df["pressure_invalid"] = df["pressure_kpa"] < 0
df.loc[df["pressure_invalid"], "pressure_kpa"] = np.nan

# Compute robust flow limits using quartiles.
q1 = df["flow_lpm"].quantile(0.25)
q3 = df["flow_lpm"].quantile(0.75)
iqr = q3 - q1

# Cap statistical flow outliers transparently.
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr
df["flow_capped"] = df["flow_lpm"].clip(lower, upper)

# Record which flow values were changed.
df["flow_was_capped"] = df["flow_lpm"] != df["flow_capped"]

# Print a compact cleaning summary.
print("Rows:", len(df))
print("Invalid pressures replaced:", int(df["pressure_invalid"].sum()))
print("Flow values capped:", int(df["flow_was_capped"].sum()))
print("Flow cap limits:", round(lower, 2), "to", round(upper, 2))

# Show only rows affected by cleaning.
changed = df[df["pressure_invalid"] | df["flow_was_capped"]]
print(changed[["time_s", "pressure_original", "pressure_kpa", "flow_original", "flow_capped"]])

# Plot original and cleaned flow readings.
plt.figure(figsize=(7, 4))
plt.plot(df["time_s"], df["flow_original"], "o--", label="Original flow")
plt.plot(df["time_s"], df["flow_capped"], "s-", label="Capped flow")
plt.xlabel("Time (s)")
plt.ylabel("Flow rate (L/min)")
plt.title("Outlier handling with traceable flags")
plt.legend()
plt.grid(True)
plt.show()



### **2.3. Time Series Cleaning**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Mechanical Engineers/Module_02/Lecture_A/image_02_03.jpg?v=1783658994" width="250">



>* Time order gives sensor data meaning
>* Build a trustworthy, aligned time index

>* Handle gaps and duplicates by context
>* Resample transparently with reproducible pandas operations

>* Smooth artifacts without hiding real events
>* Align signals and document cleaning choices



In [ ]:
#@title Python Code - Time Series Cleaning

# Time series cleaning protects engineering signal quality.
# We inspect timestamps before changing measured values.
# Each cleaning decision should remain traceable.

# pandas and numpy are available in Colab.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Create a tiny pump flow dataset.
times = pd.to_datetime([
    "2026-01-01 00:00:00", "2026-01-01 00:00:01",
    "2026-01-01 00:00:01", "2026-01-01 00:00:03",
])

# Include one impossible spike value.
flow_gpm = [50.0, 51.0, 51.0, 999.0]
raw = pd.DataFrame({"time": times, "flow_gpm": flow_gpm})

# Sort timestamps before time based operations.
raw = raw.sort_values("time").reset_index(drop=True)
raw = raw.set_index("time")

# Remove duplicate timestamps using the mean.
deduped = raw.groupby(level=0).mean()
expected = pd.date_range(deduped.index.min(), deduped.index.max(), freq="1s")

# Reindex to reveal missing seconds explicitly.
aligned = deduped.reindex(expected)
aligned.index.name = "time"

# Flag physically unrealistic pump flow readings.
valid_range = aligned["flow_gpm"].between(0, 200)
cleaned = aligned.copy()
cleaned.loc[~valid_range, "flow_gpm"] = np.nan

# Interpolate only short gaps in gradual flow data.
cleaned["flow_gpm"] = cleaned["flow_gpm"].interpolate(limit=1)
cleaned["rolling_mean"] = cleaned["flow_gpm"].rolling(2, min_periods=1).mean()

# Record transparent cleaning decisions for review.
decisions = [
    "sorted timestamps", "averaged duplicate timestamp",
    "inserted missing one-second rows", "removed impossible spike",
    "interpolated one short gap", "computed two-point rolling mean",
]

# Print a compact audit summary.
print("Raw rows:", len(raw), "Clean rows:", len(cleaned))
print("Missing after alignment:", int(aligned["flow_gpm"].isna().sum()))
print("Unrealistic values removed:", int((~valid_range).sum()))
print("Decisions:", "; ".join(decisions))

# Plot raw and cleaned flow values.
plt.figure(figsize=(7, 4))
plt.plot(raw.index, raw["flow_gpm"], "o--", label="raw")
plt.plot(cleaned.index, cleaned["flow_gpm"], "s-", label="cleaned")
plt.ylabel("Pump flow rate, gallons per minute")
plt.xlabel("Timestamp")
plt.title("Transparent time series cleaning example")
plt.legend()
plt.tight_layout()
plt.show()



## **3. Traceable Cleaning Decisions**

### **3.1. Cleaning Decision Logs**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Mechanical Engineers/Module_02/Lecture_A/image_03_01.jpg?v=1783658990" width="250">



>* Record dataset changes and reasons
>* Make cleaning choices traceable for reviewers

>* Log source differences and cleaning impacts
>* Preserve judgment behind data edits

>* Record major cleaning choices for future review
>* Link changes to transparent engineering reasons



In [ ]:
#@title Python Code - Cleaning Decision Logs

# Cleaning logs make engineering edits traceable.
# Small examples can show defensible decisions.
# Pandas records changes beside cleaned data.

import pandas as pd
import numpy as np

# Create tiny raw sensor dataset.
raw_data = {
    "time_s": [0, 1, 2, 3, 4, 4],
    "rpm": [1800, 1810, -50, np.nan, 1795, 1795],
    "temp_C": [72, 73, 74, 999, 75, 75],
}

# Build dataframe from raw records.
df_raw = pd.DataFrame(raw_data)
df_clean = df_raw.copy()
decision_log = []

# Log duplicate timestamp removal.
duplicate_mask = df_clean.duplicated(subset=["time_s"], keep="first")
duplicate_count = int(duplicate_mask.sum())
df_clean = df_clean.loc[~duplicate_mask].copy()

# Store duplicate decision details.
decision_log.append({
    "issue": "duplicate timestamp",
    "rule": "kept first record",
    "affected_rows": duplicate_count,
})

# Log impossible negative speed.
negative_mask = df_clean["rpm"] < 0
negative_count = int(negative_mask.sum())
df_clean.loc[negative_mask, "rpm"] = np.nan

# Store physical validity decision.
decision_log.append({
    "issue": "negative rpm",
    "rule": "marked invalid as missing",
    "affected_rows": negative_count,
})

# Log temperature sensor limit.
high_temp_mask = df_clean["temp_C"] > 200
high_temp_count = int(high_temp_mask.sum())
df_clean.loc[high_temp_mask, "temp_C"] = np.nan

# Store calibration range decision.
decision_log.append({
    "issue": "temperature above calibrated range",
    "rule": "marked invalid as missing",
    "affected_rows": high_temp_count,
})

# Log short gap interpolation.
missing_before = int(df_clean["rpm"].isna().sum())
df_clean["rpm"] = df_clean["rpm"].interpolate(limit=1)
missing_after = int(df_clean["rpm"].isna().sum())

# Store imputation decision details.
decision_log.append({
    "issue": "short rpm gap",
    "rule": "linear interpolation limit one",
    "affected_rows": missing_before - missing_after,
})

# Convert log into review table.
log_df = pd.DataFrame(decision_log)
print("Raw rows:", len(df_raw), "Clean rows:", len(df_clean))
print("Remaining missing values:", int(df_clean.isna().sum().sum()))

# Show compact decision log.
print(log_df.to_string(index=False))



### **3.2. Cleaning Decision Logs**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Mechanical Engineers/Module_02/Lecture_A/image_03_02.jpg?v=1783658986" width="250">



>* Record what changed and why
>* Make cleaning steps traceable for reviewers

>* Record uncertainty from mixed engineering data sources
>* Explain cleaning choices and their assumptions

>* Enable reproducible cleaning across project updates
>* Build transparent, defensible engineering conclusions



In [ ]:
#@title Python Code - Cleaning Decision Logs

# Decision logs make cleaning traceable.
# Small engineering tables reveal common issues.
# Pandas records actions beside cleaned data.

# Import pandas for tabular cleaning.
import pandas as pd

# Create tiny raw sensor records.
raw_data = [
    {"time": "08:00", "rpm": 1800, "temp_C": 72, "gap_in": 0.020},
    {"time": "08:01", "rpm": -50, "temp_C": 73, "gap_in": 0.021},

    {"time": "08:01", "rpm": -50, "temp_C": 73, "gap_in": 0.021},
    {"time": "08:02", "rpm": None, "temp_C": 74, "gap_in": 0.019},

    {"time": "08:03", "rpm": 1810, "temp_C": 999, "gap_in": 0.020},
]

# Convert records into a dataframe.
df = pd.DataFrame(raw_data)

# Prepare an empty decision log.
log_rows = []

# Record duplicate rows before removal.
duplicate_mask = df.duplicated()
duplicate_count = int(duplicate_mask.sum())

# Remove exact duplicate measurements.
df = df.loc[~duplicate_mask].copy()

# Log the duplicate cleaning decision.
log_rows.append({"issue": "duplicate row", "action": "removed", "count": duplicate_count})

# Mark impossible negative speed values.
bad_rpm_mask = df["rpm"].lt(0)
df.loc[bad_rpm_mask, "rpm"] = pd.NA

# Log the physical rule used.
log_rows.append({"issue": "negative rpm", "action": "set missing", "count": int(bad_rpm_mask.sum())})

# Mark unrealistic furnace temperature values.
bad_temp_mask = df["temp_C"].gt(300)
df.loc[bad_temp_mask, "temp_C"] = pd.NA

# Log the sensor limit decision.
log_rows.append({"issue": "temperature above limit", "action": "set missing", "count": int(bad_temp_mask.sum())})

# Convert inch gap measurements into millimeters.
df["gap_mm"] = df["gap_in"] * 25.4

# Log the unit standardization decision.
log_rows.append({"issue": "imperial gap units", "action": "converted to millimeters", "count": len(df)})

# Build a compact decision log table.
decision_log = pd.DataFrame(log_rows)

# Print concise traceability results.
print("Cleaned rows:", len(df))
print("Missing rpm values:", int(df["rpm"].isna().sum()))
print("Missing temperature values:", int(df["temp_C"].isna().sum()))
print("Decision log summary:")
print(decision_log.to_string(index=False))



### **3.3. Defensible Cleaning Records**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Mechanical Engineers/Module_02/Lecture_A/image_03_03.jpg?v=1783658988" width="250">



>* Record what changed and why
>* Make cleaning choices understandable to reviewers

>* Record issues, fixes, scope, timing, and approval
>* Explain impactful choices to show deliberate cleaning

>* Support accountable, transparent model decisions
>* Enable reproducible updates when evidence changes



In [ ]:
#@title Python Code - Defensible Cleaning Records

# Cleaning records make engineering decisions defensible.
# This example logs each important change.
# Small data keeps the workflow transparent.

import pandas as pd
import numpy as np

# Create raw pump-test readings with known issues.
raw_data = pd.DataFrame(
    {
        "time_min": [0, 1, 2, 2, 3, 4],
        "flow_gpm": [120, 121, np.nan, 123, -5, 124],
        "pressure_psi": [50, 51, 52, 52, 900, 53],
    }
)

# Copy raw data before any cleaning.
clean_data = raw_data.copy(deep=True)
cleaning_log = []

# Record duplicated timestamp handling.
duplicate_mask = clean_data.duplicated("time_min", keep="first")
cleaning_log.append(
    {
        "issue": "duplicated timestamp",
        "rows": duplicate_mask.sum(),
        "action": "kept first reading",
        "rationale": "logger reset created repeated time",
    }
)

# Remove duplicated timestamp rows.
clean_data = clean_data.loc[~duplicate_mask].copy()

# Record missing flow interpolation.
missing_flow = clean_data["flow_gpm"].isna()
cleaning_log.append(
    {
        "issue": "missing flow_gpm",
        "rows": missing_flow.sum(),
        "action": "linear interpolation",
        "rationale": "short gap during steady pump test",
    }
)

# Interpolate the short missing flow gap.
clean_data["flow_gpm"] = clean_data["flow_gpm"].interpolate()

# Record physically unrealistic values.
unrealistic = (clean_data["flow_gpm"] < 0) | (clean_data["pressure_psi"] > 200)
cleaning_log.append(
    {
        "issue": "unrealistic engineering values",
        "rows": unrealistic.sum(),
        "action": "excluded affected rows",
        "rationale": "violates pump operating envelope",
    }
)

# Exclude rows outside engineering limits.
clean_data = clean_data.loc[~unrealistic].copy()

# Convert the log into a reviewable table.
log_table = pd.DataFrame(cleaning_log)

# Print compact evidence for traceability.
print("Raw rows:", len(raw_data))
print("Clean rows:", len(clean_data))
print("Cleaning record summary:")
print(log_table.to_string(index=False))
print("Cleaned data preview:")
print(clean_data.to_string(index=False))



# <font color="#418FDE" size="6.5" uppercase>**Cleaning Engineering Data**</font>


In this lecture, you learned to:
- Detect missing, duplicated, inconsistent, and physically unrealistic values in engineering datasets. 
- Clean tabular and time-series data using transparent pandas and NumPy operations. 
- Document data-cleaning decisions so model results remain traceable and defensible. 

In the next Lecture (Lecture B), we will go over 'Features and Splits'